
# Collaborative Filtering Movie Recommender

This notebook follows the same overall structure as `CollaborativeFiltering5.ipynb`, but it uses the provided `ratings.csv` and `movies.csv` datasets.

The main change from the original guide is that the real `movieId` and `userId` values are not guaranteed to be small continuous indexes, so this notebook maps them into matrix indexes before training.


In [ ]:

import numpy as np
import pandas as pd

# Load Provided Data 
ratings = pd.read_csv("ratings.csv")
movies_df = pd.read_csv("movies.csv")

print("Ratings data:")
display(ratings.head())

print("Movies data:")
display(movies_df.head())

print(f"Total ratings: {len(ratings)}")
print(f"Unique users: {ratings['userId'].nunique()}")
print(f"Unique movies rated: {ratings['movieId'].nunique()}")
print(f"Movies in movies.csv: {len(movies_df)}")


Ratings data:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Movies data:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


Total ratings: 100836
Unique users: 610
Unique movies rated: 9724
Movies in movies.csv: 9742


In [ ]:

# Prepare Data for Matrix Factorization 

unique_users = sorted(ratings['userId'].unique())
unique_movies = sorted(ratings['movieId'].unique())

user_to_index = {user_id: index for index, user_id in enumerate(unique_users)}
movie_to_index = {movie_id: index for index, movie_id in enumerate(unique_movies)}
index_to_user = {index: user_id for user_id, index in user_to_index.items()}
index_to_movie = {index: movie_id for movie_id, index in movie_to_index.items()}

ratings_model = ratings.copy()
ratings_model['userIndex'] = ratings_model['userId'].map(user_to_index)
ratings_model['movieIndex'] = ratings_model['movieId'].map(movie_to_index)

# Keep only the columns needed for training
ratings_train = ratings_model[['userIndex', 'movieIndex', 'rating']].copy()

users = len(unique_users)
movie_count = len(unique_movies)

print(f"Users: {users}")
print(f"Movies used for training: {movie_count}")
display(ratings_train.head())


Users: 610
Movies used for training: 9724


,userIndex,movieIndex,rating
0,0,0,4.0
1,0,2,4.0
2,0,5,4.0
3,0,43,5.0
4,0,46,5.0


In [ ]:

#  Initialize User and Movie Latent Factors 


k = 10  # number of latent factors

np.random.seed(42)  # for reproducibility
U = np.random.normal(0, 0.1, size=(users, k))
M = np.random.normal(0, 0.1, size=(k, movie_count))

# Global mean rating helps stabilize predictions on a larger real dataset
global_mean = ratings_train['rating'].mean()

#  Hyperparameters 
alpha = 0.01        # learning rate
lambda_reg = 0.05   # regularization strength
epochs = 30         # use a smaller number than the guide because this dataset is much larger

print(f"Global average rating: {global_mean:.2f}")
print(f"U shape: {U.shape}")
print(f"M shape: {M.shape}")


Global average rating: 3.50
U shape: (610, 10)
M shape: (10, 9724)


In [ ]:

print("Training Collaborative Filtering...")
print(f"Users: {users}, Movies: {movie_count}, Latent factors: {k}\n")



for epoch in range(epochs):
    total_squared_error = 0.0
    shuffled_ratings = ratings_train.sample(frac=1, random_state=epoch).reset_index(drop=True)

    for row in shuffled_ratings.itertuples(index=False):
        i = int(row.userIndex)
        j = int(row.movieIndex)
        r_ij = float(row.rating)

        # Predicted rating
        pred = global_mean + np.dot(U[i], M[:, j])

        # Error
        error = r_ij - pred

        # Save old values so both updates use the same step information
        old_U_i = U[i].copy()
        old_M_j = M[:, j].copy()

        # Update latent factors with regularization
        U[i] += alpha * (error * old_M_j - lambda_reg * old_U_i)
        M[:, j] += alpha * (error * old_U_i - lambda_reg * old_M_j)

        total_squared_error += error ** 2

    rmse = np.sqrt(total_squared_error / len(ratings_train))
    print(f"Epoch {epoch + 1:3d} | RMSE: {rmse:.4f}")

print("\nTraining completed!")


Training Collaborative Filtering...
Users: 610, Movies: 9724, Latent factors: 10

Epoch   1 | RMSE: 1.0428
Epoch   2 | RMSE: 1.0411
Epoch   3 | RMSE: 1.0394
Epoch   4 | RMSE: 1.0366
Epoch   5 | RMSE: 1.0305
Epoch   6 | RMSE: 1.0178
Epoch   7 | RMSE: 0.9966
Epoch   8 | RMSE: 0.9711
Epoch   9 | RMSE: 0.9454
Epoch  10 | RMSE: 0.9199
Epoch  11 | RMSE: 0.8956
Epoch  12 | RMSE: 0.8736
Epoch  13 | RMSE: 0.8542
Epoch  14 | RMSE: 0.8371
Epoch  15 | RMSE: 0.8219
Epoch  16 | RMSE: 0.8084
Epoch  17 | RMSE: 0.7959
Epoch  18 | RMSE: 0.7846
Epoch  19 | RMSE: 0.7741
Epoch  20 | RMSE: 0.7640
Epoch  21 | RMSE: 0.7549
Epoch  22 | RMSE: 0.7459
Epoch  23 | RMSE: 0.7380
Epoch  24 | RMSE: 0.7301
Epoch  25 | RMSE: 0.7227
Epoch  26 | RMSE: 0.7158
Epoch  27 | RMSE: 0.7091
Epoch  28 | RMSE: 0.7030
Epoch  29 | RMSE: 0.6971
Epoch  30 | RMSE: 0.6915

Training completed!


In [ ]:

# Make Predictions 
# Full predicted rating matrix: rows = users, columns = movies
predicted_ratings = global_mean + np.dot(U, M)

# Clip predictions
predicted_ratings = np.clip(predicted_ratings, 0.5, 5.0)

print("Predicted Rating Matrix Shape:")
print(predicted_ratings.shape)

print("\nSmall preview of predicted ratings:")
print(np.round(predicted_ratings[:5, :5], 2))


Predicted Rating Matrix Shape:
(610, 9724)

Small preview of predicted ratings:
[[4.62 4.27 4.62 3.87 3.95]
 [3.89 3.72 3.61 3.51 3.46]
 [2.72 2.48 1.96 4.44 3.53]
 [3.65 3.25 3.49 3.22 3.64]
 [3.78 3.48 3.5  2.92 3.  ]]


In [ ]:

#  Show Original vs Predicted 
print("=" * 70)
print("Original Ratings vs Predicted Ratings")
print("Showing first 20 ratings only so the output is not too large")
print("=" * 70)

for row in ratings_model.head(20).itertuples(index=False):
    user_id = int(row.userId)
    movie_id = int(row.movieId)
    i = int(row.userIndex)
    j = int(row.movieIndex)
    true_rating = float(row.rating)
    pred_rating = predicted_ratings[i, j]
    title = movies_df.loc[movies_df['movieId'] == movie_id, 'title'].values
    title = title[0] if len(title) > 0 else f"Movie {movie_id}"

    print(
        f"User {user_id} | {title} → "
        f"True: {true_rating:.1f} | Predicted: {pred_rating:.2f} | "
        f"Error: {abs(true_rating - pred_rating):.2f}"
    )


Original Ratings vs Predicted Ratings
Showing first 20 ratings only so the output is not too large
User 1 | Toy Story (1995) → True: 4.0 | Predicted: 4.62 | Error: 0.62
User 1 | Grumpier Old Men (1995) → True: 4.0 | Predicted: 4.62 | Error: 0.62
User 1 | Heat (1995) → True: 4.0 | Predicted: 4.33 | Error: 0.33
User 1 | Seven (a.k.a. Se7en) (1995) → True: 5.0 | Predicted: 4.67 | Error: 0.33
User 1 | Usual Suspects, The (1995) → True: 5.0 | Predicted: 4.71 | Error: 0.29
User 1 | From Dusk Till Dawn (1996) → True: 3.0 | Predicted: 3.71 | Error: 0.71
User 1 | Bottle Rocket (1996) → True: 5.0 | Predicted: 4.33 | Error: 0.67
User 1 | Braveheart (1995) → True: 4.0 | Predicted: 4.84 | Error: 0.84
User 1 | Rob Roy (1995) → True: 5.0 | Predicted: 4.28 | Error: 0.72
User 1 | Canadian Bacon (1995) → True: 5.0 | Predicted: 4.40 | Error: 0.60
User 1 | Desperado (1995) → True: 5.0 | Predicted: 4.52 | Error: 0.48
User 1 | Billy Madison (1995) → True: 5.0 | Predicted: 4.26 | Error: 0.74
User 1 | Clerks 

In [ ]:

# Recommend Movies for a User 
def recommend_movies(user_id, top_n=10):
    """
    Recommend movies for a user based on predicted ratings.
    The user_id should be an original userId from ratings.csv, not the mapped userIndex.
    """
    if user_id not in user_to_index:
        print(f"User {user_id} was not found in ratings.csv")
        return pd.DataFrame()

    user_index = user_to_index[user_id]
    user_predictions = predicted_ratings[user_index]

    # Movies already rated by this user
    rated_movie_ids = set(ratings.loc[ratings['userId'] == user_id, 'movieId'].unique())

    recommendations = []
    for movie_index, predicted_rating in enumerate(user_predictions):
        movie_id = index_to_movie[movie_index]

        # Only recommend movies the user has not already rated
        if movie_id not in rated_movie_ids:
            movie_info = movies_df[movies_df['movieId'] == movie_id]
            if not movie_info.empty:
                title = movie_info.iloc[0]['title']
                genres = movie_info.iloc[0]['genres']
            else:
                title = f"Movie {movie_id}"
                genres = "Unknown"

            recommendations.append({
                'movieId': movie_id,
                'title': title,
                'genres': genres,
                'predicted_rating': predicted_rating
            })

    recommendations_df = pd.DataFrame(recommendations)
    recommendations_df = recommendations_df.sort_values(by='predicted_rating', ascending=False).head(top_n)
    recommendations_df['predicted_rating'] = recommendations_df['predicted_rating'].round(2)

    return recommendations_df.reset_index(drop=True)

# Example recommendations
recommend_movies(user_id=1, top_n=10)


,movieId,title,genres,predicted_rating
0,149352,Daddy's Home (2015),Comedy,5.0
1,1148,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime,5.0
2,7669,Pride and Prejudice (1995),Drama|Romance,5.0
3,92420,Chronicle (2012),Action|Sci-Fi|Thriller,5.0
4,82852,Little Fockers (2010),Comedy,5.0
5,7371,Dogville (2003),Drama|Mystery|Thriller,5.0
6,91094,"Muppets, The (2011)",Children|Comedy|Musical,5.0
7,1234,"Sting, The (1973)",Comedy|Crime,5.0
8,319,Shallow Grave (1994),Comedy|Drama|Thriller,5.0
9,1272,Patton (1970),Drama|War,5.0


In [10]:

# Try recommendations for another user
recommend_movies(user_id=25, top_n=10)


,movieId,title,genres,predicted_rating
0,4011,Snatch (2000),Comedy|Crime|Thriller,5.0
1,3451,Guess Who's Coming to Dinner (1967),Drama,5.0
2,3000,Princess Mononoke (Mononoke-hime) (1997),Action|Adventure|Animation|Drama|Fantasy,5.0
3,92535,Louis C.K.: Live at the Beacon Theater (2011),Comedy,5.0
4,115713,Ex Machina (2015),Drama|Sci-Fi|Thriller,5.0
5,2959,Fight Club (1999),Action|Crime|Drama|Thriller,5.0
6,7669,Pride and Prejudice (1995),Drama|Romance,5.0
7,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,5.0
8,318,"Shawshank Redemption, The (1994)",Crime|Drama,5.0
9,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,5.0


In [ ]:

#
# Change the user_id if you want recommendations for a different user.

example_user_id = 1
user_recommendations = recommend_movies(user_id=example_user_id, top_n=20)
user_recommendations.to_csv(f"recommendations_user_{example_user_id}.csv", index=False)

print(f"Saved recommendations_user_{example_user_id}.csv")
display(user_recommendations)


Saved recommendations_user_1.csv


,movieId,title,genres,predicted_rating
0,149352,Daddy's Home (2015),Comedy,5.00
1,1148,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime,5.00
2,7669,Pride and Prejudice (1995),Drama|Romance,5.00
3,92420,Chronicle (2012),Action|Sci-Fi|Thriller,5.00
4,82852,Little Fockers (2010),Comedy,5.00
5,7371,Dogville (2003),Drama|Mystery|Thriller,5.00
6,91094,"Muppets, The (2011)",Children|Comedy|Musical,5.00
7,1234,"Sting, The (1973)",Comedy|Crime,5.00
8,319,Shallow Grave (1994),Comedy|Drama|Thriller,5.00
9,1272,Patton (1970),Drama|War,5.00
